# RAG From Scratch: Agentic RAG with LangGraph

Agentic RAG adds a control loop around retrieval. The system can decide to retrieve, grade whether retrieved context is sufficient, rewrite the query, retry, and then generate an answer.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
from typing import TypedDict

from langgraph.graph import END, StateGraph
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Hybrid retrieval combines BM25 and dense semantic retrieval.",
    "Agentic RAG can rewrite queries when retrieved context is weak.",
    "RAG evaluation should measure retrieval quality and generation groundedness.",
]

vectorizer = TfidfVectorizer().fit(documents)
doc_matrix = vectorizer.transform(documents)

def retrieve(query, k=2):
    scores = cosine_similarity(vectorizer.transform([query]), doc_matrix).ravel()
    ranked = sorted(zip(scores, documents), reverse=True)
    return [doc for _, doc in ranked[:k]]

In [ ]:
class RAGState(TypedDict):
    question: str
    query: str
    docs: list[str]
    answer: str
    retries: int
    sufficient: bool

def retrieve_node(state: RAGState):
    docs = retrieve(state["query"])
    return {**state, "docs": docs}

def grade_node(state: RAGState):
    query_terms = set(state["query"].lower().split())
    context_terms = set(" ".join(state["docs"]).lower().replace(".", "").split())
    sufficient = len(query_terms & context_terms) >= 2
    return {**state, "sufficient": sufficient}

def rewrite_node(state: RAGState):
    rewritten = state["query"] + " retrieval evaluation groundedness hybrid"
    return {**state, "query": rewritten, "retries": state["retries"] + 1}

def generate_node(state: RAGState):
    context = " ".join(state["docs"])
    answer = f"Based on retrieved context: {context}"
    return {**state, "answer": answer}

def route_after_grade(state: RAGState):
    if state["sufficient"] or state["retries"] >= 1:
        return "generate"
    return "rewrite"

In [ ]:
graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_node)
graph.add_node("rewrite", rewrite_node)
graph.add_node("generate", generate_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "grade")
graph.add_conditional_edges("grade", route_after_grade, {"generate": "generate", "rewrite": "rewrite"})
graph.add_edge("rewrite", "retrieve")
graph.add_edge("generate", END)

agentic_rag = graph.compile()

agentic_rag.invoke({
    "question": "How should I improve a weak RAG answer?",
    "query": "weak answer",
    "docs": [],
    "answer": "",
    "retries": 0,
    "sufficient": False,
})

Replace the toy grader with an LLM structured output when you want production behavior: `needs_retrieval`, `context_sufficient`, `answer_grounded`, and `retry_reason` are the usual fields.